In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece tqdm

In [2]:
!unzip -q -o tika-core.zip -d /content/

In [3]:
import os
import torch
import json
from pathlib import Path
from collections import defaultdict
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import gc
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [5]:
alpha=0.2
k=10
source_files = Path("/content/tika-core/src/main/java")
rsf_path = Path("/content/tikadp-v1_ARC_10_clusters_alpha=0.2.rsf")
output_dir = Path(f"/content/week4_summary_alpha_{alpha}_k_{k}")
output_dir.mkdir(exist_ok=True)

In [6]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

In [7]:
def get_clusters(rsf_file):
    """Read ARC .rsf file and group classes by cluster ID."""
    clusters = defaultdict(list)

    if not Path(rsf_file).exists():
        print(f"ERROR: RSF file not found: {rsf_file}")
        return clusters

    with open(rsf_file, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3 and parts[0] == "contain":
                cluster_id = parts[1]
                class_name = parts[2]
                clusters[cluster_id].append(class_name)

    return clusters

def get_source_file(full_class_name, source_root):
    parts = full_class_name.split(".")
    parts[-1] = parts[-1].split("$")[0]  # handle inner classes
    rel_path = Path(*parts).with_suffix(".java")

    full_path = source_root / rel_path
    if full_path.exists() and full_path.is_file():
        try:
            return full_path.read_text(encoding="utf-8", errors="replace"), str(full_path)
        except Exception as e:
            print(f"Error reading {full_path}: {e}")
            return None, None

    return None, None


model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"\nLoading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    token=hf_token
)

def generate_summary(system_prompt, user_content, max_tokens=512):
    """Run Qwen generation with your chosen parameters."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.5,
            top_p=0.8,
            do_sample=True
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, outputs)
    ]

    del inputs
    del outputs
    torch.cuda.empty_cache()
    gc.collect()

    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


FILE_SYSTEM_PROMPT = """
You are an expert software architect analyzing Java source code.
Generate a concise structured summary with these headings:
1. Key Functionality
2. Core Logic
3. Inputs/Outputs
4. Dependencies
5. Role in Apache Tika Detect/Parser subsystem

Keep the summary factual and grounded in the provided code only.
"""

PACKAGE_SYSTEM_PROMPT = """
You are an expert software architect.
Based ONLY on the provided file summaries, generate a package/submodule summary with:
1. Package/Submodule Role
2. Shared Responsibilities
3. Internal Interactions
4. Important Dependencies

Do not invent details not supported by the summaries.
"""

CLUSTER_SYSTEM_PROMPT = """
You are an expert software architect.
Based ONLY on the provided package/module summaries, generate a cluster-level architecture summary with:
1. Architectural Title
2. High-Level Description
3. Main Responsibilities
4. Component Interactions
5. Likely Architectural Role in Apache Tika

Do not use raw code. Use only the provided summaries.
"""

if __name__ == "__main__":
  print(f"\nStarting summarization for: {rsf_path.name}")
  clusters = get_clusters(rsf_path)
  print(f"Total clusters found: {len(clusters)}")
  all_reports = {}
  missing_files = []

  for cluster_id, class_list in clusters.items():
      print(f"\n--- Processing Cluster {cluster_id} ({len(class_list)} classes) ---")

      file_summaries = {}
      package_groups = defaultdict(list)

      # ------------------------------------------
      # STEP A: FILE-LEVEL SUMMARIES
      # ------------------------------------------
      for cls_name in tqdm(class_list, desc=f"Cluster {cluster_id} files"):
          raw_code, file_path = get_source_file(cls_name, source_files)

          if raw_code is None:
              print(f"  [Skip] Source file not found for {cls_name}")
              missing_files.append(cls_name)
              continue

          truncated_code = raw_code[:4500]   # simple safety truncation

          user_prompt = f"Analyze this Java class: {cls_name}\nFile path: {file_path}\n\n```java\n{truncated_code}\n```"
          summary = generate_summary(FILE_SYSTEM_PROMPT, user_prompt, max_tokens=500)

          file_summaries[cls_name] = {
              "file_path": file_path,
              "summary": summary
          }

          package_name = ".".join(cls_name.split(".")[:-1])
          package_groups[package_name].append(f"Class: {cls_name}\nSummary:\n{summary}\n")

      # ------------------------------------------
      # STEP B: PACKAGE-LEVEL SUMMARIES
      # ------------------------------------------
      package_summaries = {}

      for pkg_name, summaries in package_groups.items():
          print(f"  [Package] Summarizing package: {pkg_name} ({len(summaries)} files)")

          if len(summaries) == 1:
              package_summaries[pkg_name] = summaries[0]
              continue

          pkg_text = "\n".join(summaries)[:9000]

          user_prompt = f"Here are file summaries for the Java package/submodule '{pkg_name}':\n\n{pkg_text}"
          pkg_summary = generate_summary(PACKAGE_SYSTEM_PROMPT, user_prompt, max_tokens=600)

          package_summaries[pkg_name] = pkg_summary

      # ------------------------------------------
      # STEP C: CLUSTER-LEVEL SUMMARY
      # ------------------------------------------
      print(f"  [Cluster] Generating final architecture summary for Cluster {cluster_id}")

      aggregated_packages = ""
      for pkg_name, pkg_summary in package_summaries.items():
          aggregated_packages += f"\nPackage: {pkg_name}\nSummary:\n{pkg_summary}\n"

      aggregated_packages = aggregated_packages[:11000]

      user_prompt = f"Here are the package/module summaries for Cluster {cluster_id}:\n\n{aggregated_packages}"
      cluster_summary = generate_summary(CLUSTER_SYSTEM_PROMPT, user_prompt, max_tokens=900)

      cluster_report = {
          "cluster_id": cluster_id,
          "classes": class_list,
          "file_summaries": file_summaries,
          "package_summaries": package_summaries,
          "architecture_summary": cluster_summary
      }

      all_reports[cluster_id] = cluster_report

      # Save per-cluster immediately
      with open(output_dir / f"cluster_{cluster_id}_summary.json", "w", encoding="utf-8") as f:
          json.dump(cluster_report, f, indent=2, ensure_ascii=False)

  with open(output_dir / "missing_files.json", "w", encoding="utf-8") as f:
      json.dump(missing_files, f, indent=2, ensure_ascii=False)

  # Save combined results
  with open(output_dir / "all_cluster_reports.json", "w", encoding="utf-8") as f:
      json.dump(all_reports, f, indent=2, ensure_ascii=False)

  print("\n✅ Week 4 summarization complete.")
  print("Saved results to:", output_dir)

  zip_name = f"/content/week4_summary_alpha_{alpha}_k_{k}.zip"
  !zip -q -r "{zip_name}" "{output_dir}"

  from google.colab import files
  files.download(zip_name)


Loading model: Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Starting summarization for: tikadp-v1_ARC_10_clusters_alpha=0.2.rsf
Total clusters found: 10

--- Processing Cluster 2 (26 classes) ---


Cluster 2 files:  12%|█▏        | 3/26 [01:52<14:24, 37.57s/it]

  [Skip] Source file not found for org.apache.tika.config.TikaComponent


...

  [Package] Summarizing package: org.apache.tika.extractor (1 files)
  [Package] Summarizing package: org.apache.tika.sax (8 files)
  [Cluster] Generating final architecture summary for Cluster 10

--- Processing Cluster 3 (2 classes) ---


Cluster 3 files: 100%|██████████| 2/2 [01:00<00:00, 30.36s/it]


  [Package] Summarizing package: org.apache.tika.metadata.filter (1 files)
  [Package] Summarizing package: org.apache.tika.mime (1 files)
  [Cluster] Generating final architecture summary for Cluster 3

✅ Week 4 summarization complete.
Saved results to: /content/week4_summary_alpha_0.2_k_10


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>